In [1]:
#1st time use
%load_ext cudf.pandas
#subsequent use
%reload_ext cudf.pandas

import yaml
import pandas as pd
import numba
import re
import string
import contractions
from textblob import TextBlob
import emoji
import tqdm
from tqdm import tqdm
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from symspellpy import SymSpell, Verbosity
import pkg_resources

[nltk_data] Downloading package stopwords to /home/dex/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
/tmp/ipykernel_3735/946472743.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
#df = cudf.read_csv('datasets/Quora Questions Pair Dataset/train.csv')
df = pd.read_csv('datasets/Quora Questions Pair Dataset/01_train.csv')
type(df)

pandas.core.frame.DataFrame

In [3]:
# remove unnecessary columns
df.drop(columns=['id','qid1','qid2'], inplace=True)

In [4]:
df.sample(3)

,question1,question2,is_duplicate
89727,I am Brahmin. My friend is Maratha. He's been ...,I'm a Brahmin girl married to a non-Brahmin wh...,0
280713,What does it feel to see your spouse in a sex ...,Is there any behind the scenes footage of sex ...,0
318346,Is it possible to play music videos continuous...,What is the best video player to watch FLV vid...,0


In [5]:
#lower case
df['question1'] = df['question1'].str.lower()
df['question2'] = df['question2'].str.lower()

In [6]:
# remove html tags
df['question1'] = df['question1'].str.replace(r'<.*?>',' ',regex=True)
df['question2'] = df['question2'].str.replace(r'<.*?>',' ',regex=True)

In [7]:
# remove web links
df['question1'] = df['question1'].str.replace(r'http\S+|www\.\S+',' ',regex=True)
df['question2'] = df['question2'].str.replace(r'http\S+|www\.\S+',' ',regex=True)

In [8]:
# expand contractions: you're --> you are, i'm --> i am
question1 = df['question1']
question2 = df['question2']
for contraction, expanded in tqdm(contractions.contractions_dict.items(), desc="Expanding Contractions..."):
    question1 = question1.str.replace(rf'\b{re.escape(contraction.lower())}\b', expanded.lower(), regex=True)
    question2 = question2.str.replace(rf'\b{re.escape(contraction.lower())}\b', expanded.lower(), regex=True)
df['question1'] = question1
df['question2'] = question2

Expanding Contractions...: 100%|████████████████████████████████████████████████████| 344/344 [00:13<00:00, 24.89it/s]


In [9]:
# expand chatwords
with open("datasets/chat_words.yaml", "r", encoding="utf-8") as f:
    chat_words = yaml.safe_load(f)

question1 = df['question1']
question2 = df['question2']
for chat_word, full_word in tqdm(chat_words.items(), desc="Correcting Questions"):
    question1 = question1.str.replace(rf'\b{re.escape(chat_word.lower())}\b', full_word, regex=True)
    question2 = question2.str.replace(rf'\b{re.escape(chat_word.lower())}\b', full_word, regex=True)
df['question1'] = question1
df['question2'] = question2

Correcting Questions: 100%|█████████████████████████████████████████████████████████| 106/106 [00:04<00:00, 25.24it/s]


In [10]:
sym_replace = {
    '%': ' percent ',
    '$': ' dollar ',
    '₹': ' rupee ',
    '₹': ' rupee ',
    '€': ' euro ',
    '@': ' at '
}
q1 = df['question1']
q2 = df['question2']
for sym, replacement in tqdm(sym_replace.items(), desc="replacing symbols..."):
    q1 = q1.str.replace(rf'\b{re.escape(sym)}\b', replacement, regex=True)
    q2 = q2.str.replace(rf'\b{re.escape(sym)}\b', replacement, regex=True)
df['question1'] = q1
df['question2'] = q2

replacing symbols...: 100%|█████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 25.69it/s]


In [27]:
# remove stopwords
stop_words = stopwords.words('english')
stopword_removed_q1 = df['question1']
stopword_removed_q2 = df['question2']
for stop_word in tqdm(stop_words, desc='Removing Stop Words'):
    stopword_removed_q1 = stopword_removed_q1.str.replace(rf'\b{re.escape(stop_word)}\b', ' ', regex=True)
    stopword_removed_q2 = stopword_removed_q2.str.replace(rf'\b{re.escape(stop_word)}\b', ' ', regex=True)
df['question1'] = stopword_removed_q1
df['question2'] = stopword_removed_q2

Removing Stop Words: 100%|██████████████████████████████████████████████████████████| 198/198 [00:07<00:00, 28.12it/s]


In [11]:
# remove emojis
emoji_pattern = r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF\U00002702-\U000027B0\U000024C2-\U0001F251]+'
df['question1'] = df['question1'].str.replace(emoji_pattern, ' ', regex=True)
df['question2'] = df['question2'].str.replace(emoji_pattern, ' ', regex=True)

In [12]:
# remove punctuations
df['question1'] = df['question1'].str.replace(r'''[!"#$%&\'()*+,\-./:;<=>?@\[\\\]^_`{|}~]''',' ',regex=True)
df['question2'] = df['question2'].str.replace(r'''[!"#$%&\'()*+,\-./:;<=>?@\[\\\]^_`{|}~]''',' ',regex=True)

In [13]:
# remove extra white spaces
df['question1'] = df['question1'].str.strip().str.replace(r'\s+',' ',regex=True)
df['question2'] = df['question2'].str.strip().str.replace(r'\s+',' ',regex=True)

In [14]:
# correct spellings using symspellpy
tokens_series1 = df['question1'].str.findall(r'\b[a-z]+\b')
tokens_series2 = df['question2'].str.findall(r'\b[a-z]+\b')
tokens_series = pd.concat([tokens_series1,tokens_series2], axis=0, ignore_index=True) #tokenization per row

all_words = tokens_series.explode().dropna()   # all unique words
unique_words = all_words.unique()#.to_pandas()  # CPU transfer

sym = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
dict_path = pkg_resources.resource_filename(
    "symspellpy", "frequency_dictionary_en_82_765.txt"
)
sym.load_dictionary(dict_path, term_index=0, count_index=1)

# creating spell correction map (dictionary) for current data
correction_map = {}
for word in unique_words:
    if word not in sym.words:  # word not in dictionary → likely misspelled
        suggestions = sym.lookup(word, Verbosity.CLOSEST, max_edit_distance=2)
        if suggestions:
            correction_map[word] = suggestions[0].term

# correcting the spellings
corrected_q1 = df['question1']
corrected_q2 = df['question2']
for wrong, right in tqdm(correction_map.items(), desc="Correcting Questions"):
    corrected_q1 = corrected_q1.str.replace(rf'\b{re.escape(wrong)}\b', right, regex=True)
    corrected_q2 = corrected_q2.str.replace(rf'\b{re.escape(wrong)}\b', right, regex=True)
df['question1'] = corrected_q1
df['question2'] = corrected_q2

Correcting Questions: 100%|█████████████████████████████████████████████████████| 29503/29503 [19:41<00:00, 24.98it/s]


In [15]:
df.isnull().sum()

question1       0
question2       2
is_duplicate    0
dtype: int64

In [16]:
df.dropna(inplace=True)
df.isnull().sum()

question1       0
question2       0
is_duplicate    0
dtype: int64

In [17]:
df['len1'] = df['question1'].str.split().list.len()
df['len2'] = df['question2'].str.split().list.len()
df.shape

(404288, 5)

In [18]:
df = df[(df['len1'] > 2) & (df['len2'] > 2)]
df.shape

(403968, 5)

In [19]:
df

,question1,question2,is_duplicate,len1,len2
0,what is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...,0,14,12
1,what is the story of kohinoor kos i door diamond,what would happen if the indian government sto...,0,10,15
2,how can i increase the speed of my internet co...,how can internet speed be increased by hacking...,0,14,10
3,why am i mentally very lonely how can i solve it,find the remainder when match match is divided by,0,11,9
4,which one dissolve in water quickly sugar salt...,which fish would survive in salt water,0,13,7
...,...,...,...,...,...
404285,how many keywords are there in the racket prog...,how many keywords are there in perl programmin...,0,14,13
404286,do you believe there is life after death,is it true that there is life after death,1,8,9
404287,what is one coin,what is this coin,0,4,4
404288,what is the approx annual cost of living while...,i am having little hairball problem but i want...,0,17,25


In [20]:
df = df[['question1', 'question2', 'is_duplicate']]

In [21]:
df.to_pandas().to_csv('datasets/Quora Questions Pair Dataset/02_train_preprocessed.csv', index=False)